Our redoing of sample_sncosmo_from_db using the more consistent class definiion.
We wish to pick of of the non-SN Ia narrow classes and fit the sncosmo models to this and store results.
Final selections transfered from original sample_sncosmo_from_db

In [ ]:
# Which class id to run here - will number for 0 to 17
classid = 14

In [ ]:
nclasses = [
    'SLSN-II', 'SLSN-I', 'SN Ia-CSM', 'SN Iax','SN Ia-SC', 'SN Ia-91T', 
    'SN Ic-BL', 'SN Ib', 'SN Ib/c', 'SN IIn', 'SN Ic', 'SN Ia-91bg',
    'SN IIP', 'SN Ia-pec', 'SN II', 'SN IIb', 'SN Ibn']

In [ ]:
print('doing fits for', nclasses[classid])

In [ ]:
# Pipeline parameters - we consider these fixed now
include_sigma = 3
plotdir = '/Users/jnordin/tmp/sncosmosample/'  # Set to None to avoid plot
#plotdir = None
classfile = '/Users/jnordin/data/models/sncosmo/sncosmo_timeseriesmodels.csv'
outdir = '/Users/jnordin/data/models/sncosmo/'
logfile = "v2_I_btssncosmo.log"

# Parameters for sncosmo model investigation
min_tot = 6.        # Min total nbr of det
min_early = 1.       # Min early nbr of det (within 15 first days)
earlytime = 15
min_bands = 2        # Min nbr of bands

# So default tested in sample_sncosmo
phaserange = 60
intdisp = 0.10
fracdisp = True
peak_chicut = 100.0     # This was changed when running this notebook to reflect where we believe warptemplates can be created 

# Parameters for peak fitting
gp_length_scale = 10.0
peakflux_iter = 0

# If the difference between the peak phase predicted by the gp and fitted by the model is less than this, marek peak as good
peak_gp_maxdiff = 15

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import json
import pymongo
import sncosmo
from datetime import datetime
from astropy.cosmology import Planck13 as cosmo
sys.path.append('/Users/jnordin/github/ampelFeb25')
from sncosmo.fitting import DataQualityError
from ampel.ztf.util.ZTFIdMapper import ZTFIdMapper
from ampel.ztf.view.ZTFFPTabulator import ZTFFPTabulator 

In [ ]:
from warpTemplate import add_warpclasses
from peakfitting_gp import estimate_peak_flux_multiband 
from peakfitting_gp import plot_peak_results_multiband
from peakfitting_gp import get_peak_colors

In [ ]:
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')
df = add_warpclasses(df_bts, purge=True)
print(df.shape)

In [ ]:
# Limit to our set
df = df[(df['type_n']==nclasses[classid]) & ( df['redshift']!='-')]
df['redshift'] = pd.to_numeric(df['redshift'])
print(df.shape)

In [ ]:
df_class = pd.read_csv(classfile,sep=';')

In [ ]:
tabulators = []
tabulators.append(
    ZTFFPTabulator(inclusion_sigma=include_sigma)    
)

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase

In [ ]:
# Incorporate results from the students
# The inspection of fits by suleyman based on the first list
suleyman_reject = [
    'ZTF18acwyvet', 'ZTF18adbikdz', 'ZTF19aadnxbh', 'ZTF19acxxwrs', 'ZTF20aalrqbu', 
    'ZTF20aamttiw', 'ZTF20aazycgy', 'ZTF20abcjdwu', 'ZTF20abefbpl', 'ZTF20accmutv', 
    'ZTF20aciwcuz', 'ZTF21aabpnlm', 'ZTF21acadrzr', 'ZTF22aaaefgq'
]

### Setting things up
Help methods and sncosmo model definition

In [ ]:
def get_db_table( name, database, tabulators=[] ):
    """
    For ZTF name, get photopoints and then tables. 
    """

    # Name
    if isinstance(name, int):    
        print('Assuming name given as stock')
        stock = int
    elif re.search('ZTF', name):
        stock = ZTFIdMapper.to_ampel_id(name)
    else:
        print('Cannot parse', name)
        return None 

    # Obtain photopoints
    dps = [dp for dp in database.t0.find({'stock':stock})]

    # Convert to table(s)
    ftables = []
    for tabulator in tabulators:
        ftables.append( tabulator.get_flux_table(dps) )
    if len(ftables)>1:
        print('Implement astropy table appending!')
    return ftables.pop(0)

In [ ]:
import numpy as np
from astropy.table import Table
import sncosmo


def deredden_flux_table(table, A_V, R_V=3.1):
    """
    Correct a photometry table for Milky Way extinction.

    Parameters
    ----------
    table : astropy.table.Table
        Must contain columns:
        'flux', 'fluxerr', and 'band'.

    A_V : float
        Milky Way extinction in magnitudes.

    R_V : float, optional
        Ratio of total to selective extinction.
        Default is 3.1.

    Returns
    -------
    astropy.table.Table
        Copy of the input table with dereddened
        flux and fluxerr columns.
    """
    result = table.copy()

    dust = sncosmo.CCM89Dust()
    dust.set(ebv=A_V / R_V)

    # Compute correction factor for each unique band
    corrections = {}

    for band in np.unique(result["band"]):
        bp = sncosmo.get_bandpass(band)

        trans = dust.propagate(
            bp.wave,
            np.ones_like(bp.wave)
        )

        # Bandpass-averaged transmission
        T_eff = (
            np.trapz(trans * bp.trans, bp.wave)
            / np.trapz(bp.trans, bp.wave)
        )

        corrections[band] = 1.0 / T_eff

    # Apply corrections
    factors = np.array(
        [corrections[b] for b in result["band"]]
    )

    result["flux"] *= factors
    result["fluxerr"] *= factors

    return result

In [ ]:
models = {}

In [ ]:
# Vanilla salt2
models['salt2'] = {
    'model': sncosmo.Model(source='salt2'), 
    'fitprop': ['t0', 'x0', 'x1', 'c'],
    'bounds': None,
    'class': 'SN Ia'
}
peakphase = {
            'ztfg':models['salt2']['model'].source.peakphase('ztfg'),
            'ztfr':models['salt2']['model'].source.peakphase('ztfr'),
        }
peakphase['avg'] = (peakphase['ztfg']+peakphase['ztfr'])/2
models['salt2']['peakphase'] =  peakphase
# salt3
models['salt3'] = {
    'model': sncosmo.Model(source='salt3'), 
    'fitprop': ['t0', 'x0', 'x1', 'c'],
    'bounds': None,
    'class': 'SN Ia'
}
peakphase = {
            'ztfg':models['salt3']['model'].source.peakphase('ztfg'),
            'ztfr':models['salt3']['model'].source.peakphase('ztfr'),
        }
peakphase['avg'] = (peakphase['ztfg']+peakphase['ztfr'])/2
models['salt3']['peakphase'] =  peakphase


In [ ]:
dust = sncosmo.CCM89Dust()

In [ ]:
tsources = {n:c for n, c in zip(df_class['Name'], df_class['Type']) }

In [ ]:
# Where do we correct for mw dust?
for tsource, sourceclass in tsources.items():
    if tsource[0]=='#':
        continue
    if tsource in ['salt2','salt3']:
        continue
    print('Adding', tsource)
    m = sncosmo.Model(source=tsource,
                      effects=[dust],
                      effect_names=['host'],
                      effect_frames=['rest'])
    m.set(hostr_v = 3.1)
    peakphase = {
            'ztfg':m.source.peakphase('ztfg'),
            'ztfr':m.source.peakphase('ztfr'),
        }
    peakphase['avg'] = (peakphase['ztfg']+peakphase['ztfr'])/2
    models[tsource] = {
        'model': m,
        'fitprop': ['t0', 'amplitude', 'hostebv'],
        'bounds': {
            'hostebv':[-3.,5.],
        },
        'class': sourceclass,
        'peakphase': peakphase,
    }

In [ ]:
#results = {modelname:[] for modelname in models.keys()}

In [ ]:
failkey = []
for k, row in df.iterrows():
    (name, z) = (row['ZTFID'],row['redshift'])
#    if not name=='ZTF19abpvbzf':
#        continue
    if k<=15000:
        continue
    #if k>18000:
    #    break
    print(k, df.shape[0], row['ZTFID'],row['redshift'])
    if name in suleyman_reject:
        print('.. bad one, skipping')
        failkey.append(1)
        continue
    tab = get_db_table(name, database=db, tabulators=tabulators)
    tab.sort('time')
    if len(tab)==0:
        failkey.append(2)
        continue
    bands = len(set(tab['band']))

    # Do not bother about single band lightcurves
    if bands<2:
        failkey.append(3)
        continue
    
    # Correct table for MW extinction 
    tab = deredden_flux_table(tab, row['A_V'], R_V=3.1)

    
    banddict = {
        band : {'time':tab[tab['band']==band]['time'], 'flux':tab[tab['band']==band]['flux'], 'flux_err':tab[tab['band']==band]['fluxerr']}
        for band in set(tab['band'])
    }
    results_gp = estimate_peak_flux_multiband(banddict, method="gp", 
                                           length_scale=gp_length_scale, 
                                           n_sigma=3,
                                           n_clip_iter=peakflux_iter,
                                          )
    peakcol =  get_peak_colors( results_gp, prefix='gp_', min_eff_points=1 )

    if not 'gp_ztfg-ztfr' in peakcol:
        print('not gp peak col')
        failkey.append(4)
        continue
    presum = peakcol.get('gp_ztfg_n_eff_before_peak',0) + peakcol.get('gp_ztfr_n_eff_before_peak',0)
    postsum = peakcol.get('gp_ztfg_n_eff_after_peak',0) + peakcol.get('gp_ztfr_n_eff_after_peak',0)
    peakerr = peakcol.get('gp_ztfg-ztfr_err',0)
    peakr = results_gp['ztfr'].peak_time
    if presum<2 or postsum<2:
        print('few gp dp')
        failkey.append(5)
        continue

    

    atleastone = 0
    for modelname, modelpar in models.items():
#        if not modelname=='snana-2006ez':
#            continue
        m = modelpar['model']
        t0guess = peakr - modelpar['peakphase']['ztfr']
        m.set(z=z)
        m.set(t0=t0guess)
        bounds = {'t0':[tab['time'].min()-5., tab['time'].max()+5.]}
        if modelpar['bounds'] is not None:
            bounds.update( modelpar['bounds'])
        # run the fit
        try:
            result, fitted_model = sncosmo.fit_lc(
                tab, m,
                modelpar['fitprop'],  # parameters of model to vary
                bounds=bounds)  # bounds on parameters (if any)
        except (RuntimeError, ValueError, KeyError, DataQualityError) as e:
            print('.. fit fail')
            # Fit failed
            results[modelname].append(
                {
                    'id':name,
                    'nbr_bands':bands,
                    'ndet':len(tab),
                    'success': False,
                }
            )
            continue

        if result.ndof==0:
            results[modelname].append(
                {
                    'id':name,
                    'nbr_bands':bands,
                    'ndet':len(tab),
                    'success': False,
                }
            )
            continue

        

        
        

        # Store summary values
        mdict = {
            result['param_names'][k]:result['parameters'][k] for k in range(len(result['parameters'])) 
        }
        mdict.update( {k:result[k] for k in ['success', 'chisq', 'ndof', 'errors']} )
        mdict['absmag'] = fitted_model.source_peakabsmag(band='bessellb',magsys='ab')
        mdict['peakmag'] = row['peakmag']
        mdict['chidof'] = result.chisq / result.ndof 
        mdict['id'] = name
        mdict['nbr_bands'] = bands
        mdict['ndet'] = len(tab)
        mdict['class'] = modelpar['class']
        mdict['presum'] = presum
        mdict['postsum'] = postsum

        # Check whether the fitted peak is very far from the gp prediction
        mdict['t0diff'] = np.abs( fitted_model.get('t0')-t0guess )

        
        # Which time-range should we consider
        tstart = fitted_model.mintime()
        tend = tstart + min(fitted_model.maxtime(), tstart+phaserange)

        # Check whether data extends largely beyond limits
        mdict['premod'] = sum( ( tab['time']<tstart ) )
        mdict['postmod'] = sum( ( tab['time']>tend ) )
        mdict['predt'] = min(tab['time'])-tstart
        mdict['postdt'] = max(tab['time'])-tend
        if mdict['premod']>1 or mdict['postmod']>1 or mdict['predt']<-3 or mdict['postdt']>3:
            continue
        
        
        peaktab = tab[
            ( tab['time']>=tstart ) & ( tab['time']<=tend )
        ]
        
        # Try to make a prediction
        try:
            fdiff = peaktab['flux'] - fitted_model.bandflux(peaktab['band'], peaktab['time'], zp=peaktab['zp'], zpsys=peaktab['zpsys'])
            if fracdisp:
                serr = np.sqrt( (peaktab['flux']*intdisp)**2 + peaktab['fluxerr']**2 )
            else:
                serr = np.sqrt( (np.mean(peaktab['flux'])*intdisp)**2 + peaktab['fluxerr']**2 )
            chi = sum( fdiff**2 / serr**2 )
            mdict['peakchi'] = chi
        except ValueError:
            mdict['peakchi'] = 1000.
            print('why fail? - prob because filter extends outside template size')
            continue


        # Check whether peak prediction compared with peak flux - they should not be too different
        peakpoint = peaktab[peaktab['flux']==peaktab['flux'].max()]
        peakdiff = ( 
            (
            peakpoint['flux']-fitted_model.bandflux(peakpoint['band'], fitted_model.maxtime(), zp=peakpoint['zp'], zpsys=peakpoint['zpsys'])
            ) / peakpoint['flux']
        )
        if peakdiff>1.5 or peakdiff<0.5:
            continue


        # Numer of detections between start and 60 days later
        mdict['peakdet'] = len(peaktab)


        mdict['peakbands'] = len(set(peaktab['band']) )
        if len(peaktab)>1:
            mdict['peakduration'] = max(peaktab['time']) - min(peaktab['time'])

        # Directly check region around mean peak:
        # Number of detections first 15 days 
        # peakphase does not seem to work ...
        fitpeaktime = modelpar['peakphase']['avg'] + mdict['t0']
#        fitpeaktime = mdict['t0']
        verypeakdt = 3
        mdict['earlydet'] = sum( peaktab['time']<= (fitpeaktime-verypeakdt) )
        mdict['thendet'] = sum( (peaktab['time']<= (fitpeaktime+verypeakdt)) & (peaktab['time']> (fitpeaktime-verypeakdt)) )
        mdict['postdet'] = sum( (peaktab['time']<= (fitpeaktime+5*verypeakdt)) & (peaktab['time']> (fitpeaktime+verypeakdt)) )
        mdict['latedet'] = sum( (peaktab['time']<= (fitpeaktime+10*verypeakdt)) & (peaktab['time']>= (fitpeaktime+5*verypeakdt)) )
        min_verypeakcount = 1
        verypeaktab = tab[
            ( tab['time']>=(fitpeaktime-10*(1+z)) ) & ( tab['time']<=(fitpeaktime+10*(1+z)) )
        ]
        mdict['verypeakcount'] = len(verypeaktab)
        mdict['verypeakbands'] = len( set(verypeaktab['band']) )

        
        # Evaluate whether fit should be considered good
        if ( 
#            (mdict['peakdet']>=min_tot) and (mdict['earlydet']>=min_early) and (mdict['peakbands']>=min_bands)
#            and (mdict['thendet']>=min_early) and (mdict['postdet']>=min_early)
#            and mdict['verypeakcount']>=min_verypeakcount 
#            and (mdict['peakchi']/mdict['peakdet'])<peak_chicut
            mdict['t0diff'] > peak_gp_maxdiff or mdict['presum'] < 3 or mdict['postsum'] < 3 
            or mdict['earlydet']<2 or mdict['postdet']<2 or mdict['latedet']<2
        ):
            mdict['peak_good'] = False
            if atleastone==0:
                atleastone = 1
        else:
            mdict['peak_good'] = True
            atleastone = 2

        results[modelname].append(mdict)
    
        plotname = '{:.2}_{}_{}_{:.2}_{:.2}_{:.2}.png'.format(presum, name, modelname, mdict['peakchi']/mdict['peakdet'], postsum, mdict['t0diff'])

            
        if plotdir is not None:
            if mdict['peak_good']:
                fout = plotdir+'goodpeak/'+plotname
            else:
                fout = plotdir+'badpeak/'+plotname

            fig = sncosmo.plot_lc(tab, model=fitted_model, errors=result.errors)
            plt.axvline(x=fitpeaktime-fitted_model.get('t0'))
#            plt.axvline(x=fitpeaktime-verypeakdt-fitted_model.get('t0'), linestyle='dotted')
#            plt.axvline(x=fitpeaktime+verypeakdt-fitted_model.get('t0'), linestyle='dotted')
            plt.savefig(fout)
            plt.close()

    if atleastone==0:
        failkey.append(6)
    elif atleastone==1:
        failkey.append(9)
    if atleastone==2:
        failkey.append(99)
        


### Make some analysis
How many sne did at least have one model fit registered as good?
How many templates did at least have one model fit registered as good?


In [ ]:
plt.hist( [f for f in failkey if f<50] )

In [ ]:
len( [f for f in failkey if f>50] )

In [ ]:
# In order to reevaluate what this means we would like to know:
# - how many sne have at least one reasonable template fit?
# - how many of these are below are noted max redshift?

# Redshift ranges for particular narrow classes if different. Should shift to class once ready?
zclass = {
    'SLSN-II': [0.0, 0.3],
    'SLSN-I': [0.0, 0.3],
    'SN Ia-91bg': [0.01, 0.055],
    'SN Ia-91T': [0.01, 0.10],
    'SN Ia-CSM': [0.01, 0.10],
    'SN IIn': [0.0, 0.10],
    'SN Ia-SC': [0.01, 0.10],
    'SN Ia-pec': [0.01, 0.055], 
    'SN Iax': [0.0, 0.055], 
}
zlim = [0.0,0.07]
if nclasses[classid] in zclass:
    zlim = zclass[nclasses[classid]]
    print('Using specific z lim.')
else:
    print('using default z lim.')

In [ ]:
# Look at good peak fits
fitstore = []
fitvolume = []
fitgood = []
for modname, reslist in results.items():
    for res in reslist:
        fitstore.append( res['id'] )
        if not res['success']:
            continue
        if not res['peak_good']:
            continue
        fitgood.append( res['id'] )
        if zlim[0] <= float(res['z']) <= zlim[1]:
            fitvolume.append( res['id'] )

In [ ]:
# Plot some redshift distributions
plt.figure(1, figsize=(10,5))
_, bins, __ = plt.hist( df['redshift'], bins=10)
plt.hist( df['redshift'].loc[df['ZTFID'].isin(fitstore)], bins=bins, label='Fit done')
plt.hist( df['redshift'].loc[df['ZTFID'].isin(fitgood)], bins=bins, label='Fit good')
plt.hist( df['redshift'].loc[df['ZTFID'].isin(fitvolume)], bins=bins, label='Fit in vol')
plt.legend()

In [ ]:
print(nclasses[classid], zlim[0], zlim[1], df.shape[0], len(set(fitstore)), len(set(fitgood)), len(set(fitvolume)))

In [ ]:
with open(logfile, "a") as f:
    timestamp = datetime.now().isoformat(timespec="seconds")
    f.write("{} {} {} {} {} {} {} {} \n".format(nclasses[classid], zlim[0], zlim[1], df.shape[0], len(set(fitstore)), len(set(fitgood)), len(set(fitvolume)), timestamp))

In [ ]:
# Store results
fname = outdir+'btsfitsv2_{}.json'.format(nclasses[classid].replace('/',''))

In [ ]:
def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    elif isinstance(obj, tuple):
        return tuple(make_json_serializable(v) for v in obj)
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

In [ ]:
jresults = make_json_serializable( results )

In [ ]:
with open(fname, "w") as outfile:
    outfile.write(json.dumps(jresults, indent=2))

In [ ]:
len( jresults )